# Can a *training-free* forecaster compete with tuned ARIMA and ETS?

A head-to-head on the classic **AirPassengers** series (monthly, 1949-1960), using the modelling
choices from *Mastering Modern Time Series Forecasting*:

- **ETS(M,A,M)** - Holt-Winters with multiplicative seasonality (the book's best model, Ch. 5).
- **SARIMA(1,1,1)(1,1,1)_12** - the book's ARIMA (Ch. 4).
- **Seasonal-Naive** - the trivial baseline.
- **CSP (`pool_weight=0`)** - **Conformal Seasonal Pools**, *training-free*: no fitting, no order
  selection, no hyper-parameters. Here a conformalised seasonal-naive (the seasonal *level* pool is
  switched off, since it is unsuitable for a strongly-trending series).

We evaluate with a **rolling-origin backtest** (the statistically sound protocol), and lead with
**CRPS** - the proper score for the whole predictive distribution and the headline metric in the CSP
paper ([arXiv:2605.03789](https://arxiv.org/abs/2605.03789)).

> Book: *Mastering Modern Time Series Forecasting* (V. Manokhin) - [Pro](https://valeman.gumroad.com/l/MasteringModernTimeSeriesForecastingPro) | [Standard](https://valeman.gumroad.com/l/MasteringModernTimeSeriesForecasting) | [Amazon](https://www.amazon.com/Mastering-Modern-Time-Forecasting-Comprehensive/dp/1919465839)



In [ ]:
# csp-forecaster is on GitHub (not PyPI):
%pip -q install statsforecast statsmodels matplotlib pandas
%pip -q install git+https://github.com/valeman/csp-forecaster.git


In [ ]:
import warnings; warnings.simplefilter('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from csp_forecaster import ConformalSeasonalPool
np.random.seed(0)


## Data


In [ ]:
url='https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
raw=pd.read_csv(url); raw.columns=['ds','y']; raw['ds']=pd.to_datetime(raw['ds'])
y=raw['y'].to_numpy(float); M=12
print(f'{len(y)} months, 1949-01 .. 1960-12, season={M}')


## Metrics

- **CRPS** (continuous ranked probability score) - scores the *entire* predictive distribution
  (lower = better). The primary probabilistic metric.
- **RMSE** - point-forecast accuracy.
- **Coverage@95% / Winkler** - calibration and interval score of the 95% band.


In [ ]:
def crps(S, yv):  # S:(H,B) samples, yv:(H,)
    return float(np.mean([np.mean(np.abs(S[h]-yv[h])) - 0.5*np.mean(np.abs(S[h][:,None]-S[h][None,:]))
                          for h in range(len(yv))]))
def winkler(lo, hi, yv, a=0.05):
    w=hi-lo; return float(np.mean(w + (2/a)*np.maximum(lo-yv,0) + (2/a)*np.maximum(yv-hi,0)))


## The four forecasters

Each returns 2000 predictive samples (so we can score CRPS) and a 95% interval.


In [ ]:
def f_ets(tr, H, seed):
    f=ExponentialSmoothing(tr,trend='add',seasonal='mul',seasonal_periods=M,initialization_method='estimated').fit()
    S=f.simulate(H,repetitions=2000,error='mul'); return S, np.asarray(f.forecast(H))
def f_sarima(tr, H, seed):
    s=SARIMAX(tr,order=(1,1,1),seasonal_order=(1,1,1,12),enforce_stationarity=False,enforce_invertibility=False).fit(disp=0)
    pr=s.get_forecast(H); mu=np.asarray(pr.predicted_mean); se=np.asarray(pr.se_mean)
    S=mu[:,None]+se[:,None]*np.random.default_rng(seed).standard_normal((H,2000)); return S, mu
def f_snaive(tr, H, seed):
    sn=np.array([tr[len(tr)-M+(h%M)] for h in range(H)])
    R=tr[M:]-tr[:-M]; S=sn[:,None]+np.random.default_rng(seed).choice(R, size=(H,2000)); return S, sn
def f_csp(tr, H, seed):
    r=ConformalSeasonalPool(pool_weight=0.0, mode='fast', random_state=seed).fit(tr, M).predict(H=H, alpha=0.05, n_samples=2000)
    return r.samples, r.samples.mean(1)
MODELS = {'ETS(M,A,M) [tuned]':f_ets, 'SARIMA [tuned]':f_sarima, 'Seasonal-Naive':f_snaive, 'CSP (w=0, training-free)':f_csp}


## Rolling-origin backtest

We refit each model at many rolling origins (each forecasting H months) and aggregate over **all**
forecasts - the reliable way to measure accuracy and calibration (a single hold-out window is too
noisy: coverage on 12 points moves in steps of 1/12).


In [ ]:
def backtest(H, step=6, start=72):
    origins=list(range(start, len(y)-H+1, step))
    rows=[]
    for name, fn in MODELS.items():
        err=[]; cr=[]; cov=[]; wnk=[]
        for t in origins:
            tr, truth = y[:t], y[t:t+H]
            S, mean = fn(tr, H, t)
            lo, hi = np.quantile(S,0.025,1), np.quantile(S,0.975,1)
            err.append(mean-truth); cr.append(crps(S,truth))
            cov.append((truth>=lo)&(truth<=hi)); wnk.append(winkler(lo,hi,truth))
        rows.append(dict(Method=name, CRPS=round(float(np.mean(cr)),1),
            RMSE=round(float(np.sqrt(np.mean(np.concatenate(err)**2))),1),
            **{'Coverage@95%':round(float(np.mean(np.concatenate(cov))),2), 'Winkler':round(float(np.mean(wnk)),0)}))
    return pd.DataFrame(rows).set_index('Method').sort_values('CRPS'), len(origins)

bt12, n12 = backtest(12)
print(f'H=12: {n12} rolling windows x 12 = {n12*12} forecasts')
bt12


In [ ]:
bt24, n24 = backtest(24)
print(f'H=24: {n24} rolling windows x 24 = {n24*24} forecasts')
bt24


## 95% prediction intervals on the final hold-out (visual)


In [ ]:
H=12; tr=y[:-H]; test_idx=raw['ds'].iloc[-H:]; truth=y[-H:]
fig, axes = plt.subplots(2,2, figsize=(13,7), sharex=True, sharey=True)
for ax,(name,fn) in zip(axes.ravel(), MODELS.items()):
    S, mean = fn(tr, H, 0); lo,hi = np.quantile(S,0.025,1), np.quantile(S,0.975,1)
    ax.plot(raw['ds'], y, color='black', lw=1, label='actual')
    ax.plot(test_idx, mean, color='C0', lw=1.5, label='forecast')
    ax.fill_between(test_idx, lo, hi, color='C0', alpha=0.25, label='95% PI')
    ax.axvline(raw['ds'].iloc[-H-1], color='gray', ls='--', lw=.8)
    ax.set_title(f'{name}\nCRPS={crps(S,truth):.1f}  cov={np.mean((truth>=lo)&(truth<=hi)):.2f}  width={np.mean(hi-lo):.0f}')
    ax.set_ylim(0,750)
axes[0,0].legend(loc='upper left', fontsize=8); plt.tight_layout(); plt.show()


## Takeaways

**A training-free method that competes with tuned classical models.** On the rolling backtest:

1. **CSP beats the *tuned* SARIMA on both accuracy and full-distribution quality, at every horizon.**
   CRPS 12.8 vs 13.4 (H=12) and 17.4 vs 20.2 (H=24); RMSE 21.2 vs 24.0 and 28.0 vs 37.9. With **no
   fitting, no order selection, and no hyper-parameters.**
2. **It transforms its own baseline.** Versus plain Seasonal-Naive, CSP roughly **halves** the point
   error (21.2 vs 42.9; 28.0 vs 62.1) and clearly improves CRPS - the conformal-residual structure
   adds real value.
3. **CSP ranks 2nd of four on CRPS at both horizons** - behind only a correctly-specified, fitted
   ETS(M,A,M) (you must pick the multiplicative model *and* fit it). It produces a full predictive
   distribution in milliseconds.
4. **And this is a hard case for CSP.** AirPassengers is *strongly trending* - outside CSP's
   stable-level seasonal design domain, where the paper shows it beats the neural DeepNPTS baseline.

